# 反问题求解入门 —— `inv_framework` 教程

> **读者**：负责搭建 Inverse-Problem Agent 框架的前端同学。默认你**数学功底扎实、熟悉优化**（梯度下降、凸性、范数、SVD），但**没接触过反问题（inverse problem, IP）的语言体系**。
>
> **目标**：用一节课的篇幅讲清楚
> 1. 反问题到底是什么，和你熟悉的「优化问题」是什么关系；
> 2. 为什么反问题「病态（ill-posed）」，以及病态性在数值上长什么样；
> 3. 常见应用场景及其**数学前向模型**（线性：CT / MRI / PET；非线性 PDE：Helmholtz / Eikonal）；
> 4. 迭代求解方法与正则化（这部分和你的优化背景一一对应）；
> 5. 重建质量评估指标；
> 6. **`inv_framework` 的每一个函数接口对应哪条公式**，以及你在 Agent 框架里应该把哪些东西作为「可调度的算子 / 求解器」暴露出去。

每一节都配有**可直接运行的 demo**。代码量不大，关键是把「公式 ↔ 接口」对上。

---

## 一句话先建立直觉

> 你已经会解 $\min_x f(x)$。**反问题求解 ≈ 给 $f$ 选一个合适的形状，然后调你已经会的优化器。**

反问题里那个「形状」由两块拼成：
- **数据保真项** `data fidelity`：由一个**前向算子 `A`** 决定（`inv_framework` 的 `ForwardOperator`）；
- **正则项 / 先验** `prior`：由你选的**求解器 `Solver`** 隐式或显式决定（`InverseProblemSolver`）。

整个框架就是把这两块解耦，让 Agent 可以「挑一个 A」配「挑一个 Solver」。

## 目录

0. [环境与导入](#sec0)
1. [反问题的定义](#sec1) — `ForwardOperator` / `NoiseModel`
2. [病态性（ill-posedness）](#sec2) — 为什么不能直接 $A^{-1}y$
3. [应用场景与前向模型](#sec3) — CT / MRI / PET / Helmholtz / Eikonal
4. [迭代优化方法与正则化](#sec4) — Landweber / SIRT / 变分 / DIP / INR / 扩散 DPS
5. [评估指标](#sec5) — PSNR / SSIM / 残差
6. [非线性 demo 与求解器分派](#sec6) — Agent 该怎么选 solver
7. [暴露给前端：如何扩展](#sec7) — 新算子 / 新求解器 / 新先验
8. [速查表：公式 ↔ 代码](#sec8)

<a id="sec0"></a>
## 0. 环境与导入

依赖：`torch>=2.0`（必需）、`matplotlib`（画图）。可选 `diffusers`（扩散类求解器换更强 UNet）、`astra-toolbox`（3D CT）。

本 notebook 默认放在仓库根目录 `inv_framework/` 下。若放在别处，改一下 `PROJECT_ROOT` 即可。

> demo 全部用 **64×64 小图 + 少量迭代**，CPU 上即可跑完（几十秒级）。真实任务请把 `image_size` 和迭代数调大并切到 GPU。

In [ ]:
import os, sys, math
import torch

# 让 notebook 能 import 到 inv_framework 包（仓库根目录）
PROJECT_ROOT = os.path.abspath('.')          # 如有需要改成 '/home/caoxiang/Desktop/IP-Agent/inv_framework'
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import matplotlib.pyplot as plt

torch.manual_seed(0)
# 教程默认用 CPU 保证任何机器可复现；有空闲 GPU 可改成 'cuda'
DEVICE = 'cpu'
print('torch', torch.__version__, '| device', DEVICE)

def show(imgs, titles, vmin=0, vmax=1, cmap='gray', figsize=None, sino_idx=()):
    '''小工具：并排显示若干 (1,1,H,W) 或 (1,1,A,W) 张量。'''
    n = len(imgs)
    fig, axes = plt.subplots(1, n, figsize=figsize or (3*n, 3.2))
    if n == 1: axes = [axes]
    for i, (ax, im, t) in enumerate(zip(axes, imgs, titles)):
        arr = im.detach()[0, 0].cpu().numpy()
        if i in sino_idx:
            ax.imshow(arr, cmap=cmap, aspect='auto')
        else:
            ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax)
        ax.set_title(t); ax.axis('off')
    plt.tight_layout(); plt.show()

<a id="sec1"></a>
## 1. 反问题的定义

### 1.1 正问题 vs 反问题

设有一个物理 / 测量过程，把我们关心的未知量 $x$（图像、介质参数场……）映射成观测 $y$：

$$\boxed{\,y = \mathcal{A}(x) + n\,}$$

- $x \in \mathcal{X}$：**待重建量**（reconstruction），如 CT 中的衰减系数图、USCT 中的声速场；
- $\mathcal{A}:\mathcal{X}\to\mathcal{Y}$：**前向算子（forward operator）**，编码物理；
- $n$：**测量噪声**；
- $y \in \mathcal{Y}$：**测量数据（measurement）**。

| | 已知 | 求 | 难度 |
|---|---|---|---|
| **正问题（forward）** | $x$, $\mathcal{A}$ | $y=\mathcal{A}(x)$ | 通常良定，直接算 |
| **反问题（inverse）** | $y$, $\mathcal{A}$ | $x$ | 通常**病态**，本教程主题 |

> 对优化背景的你：正问题是「求值」，反问题是「求逆」。当 $\mathcal{A}$ 不可逆 / 接近奇异 / 非线性时，直接求逆失败，于是退而求其次——**把它写成一个优化问题**（见第 4 节）。

### 1.2 框架里的对应物

`inv_framework` 把上式的三个对象分别抽象成三层（彻底解耦）：

| 数学对象 | 代码抽象 | 文件 |
|---|---|---|
| $\mathcal{A}$（前向算子） | `ForwardOperator` / `LinearOperator` | `operators/base.py` |
| $n$（噪声 / 似然） | `NoiseModel` | `operators/noise.py` |
| 「从 $y$ 求 $x$」的算法 | `InverseProblemSolver` | `solvers/base.py` |

`ForwardOperator` 的核心契约只有一条：

```python
class ForwardOperator(ABC):
    domain_shape: tuple        # x 单样本形状 (C,H,W)，对应定义域 X
    range_shape:  tuple        # y 单样本形状，对应值域 Y
    def forward(self, x): ...  # 实现 A(x)，且必须 autograd 可微
    def __call__(self, x): return self.forward(x)
```

> **关键约束**：`forward` 必须**对 `x` 可反传（autograd-traceable）**。原因见第 4 节——DIP / INR / 扩散类方法全靠对 $\|\mathcal{A}(x)-y\|^2$ 自动求 $\partial/\partial x$。这是把「物理」接进「优化器」的桥。

线性算子多一条 `adjoint`（即 $\mathcal{A}^{\top}$），下一节细讲。

### 1.3 噪声模型 = 似然函数

$n$ 的分布决定了数据保真项的形式（最大似然视角）：

- **高斯噪声** $n\sim\mathcal{N}(0,\sigma^2 I)$ ⟹ 负对数似然 $\propto \tfrac{1}{2\sigma^2}\|\mathcal{A}(x)-y\|_2^2$（最常见的 L2 保真项）。代码：`GaussianNoise(sigma)`。
- **泊松噪声**（光子计数，CT / PET 真实物理）⟹ 保真项是 KL / Poisson 负对数似然。代码：`PoissonLogDomainNoise(photon_count, ...)`，建模 $I=\mathrm{Poisson}(I_0 e^{-\mu x})$ 后返回 $-\log(I/I_0)$，仍落在 sinogram 域。
- `NoNoise`：无噪，调试用。

下面跑第一个 demo：构造一个目标图 $x$，用 CT 前向算子生成测量 $y$。

In [ ]:
from inv_framework.operators.ct.radon_torch import ParallelBeamRadon2D
from inv_framework.operators.noise import GaussianNoise, NoNoise

N = 64
# 造一个简单 phantom 当作 ground-truth x（一个方块 + 一个圆）
yy, xx = torch.meshgrid(torch.linspace(-1,1,N), torch.linspace(-1,1,N), indexing='ij')
x_true = torch.zeros(1,1,N,N, device=DEVICE)
x_true[..., 16:48, 16:48] = 0.5                     # 方块
x_true[0,0][(xx**2 + (yy-0.0)**2) < 0.12] = 1.0     # 圆盘
x_true = x_true.clamp(0,1)

# 前向算子 A：平行束 Radon 变换（CT），30 个投影角
A = ParallelBeamRadon2D(image_size=N, num_angles=30, device=DEVICE)
print('domain_shape (x):', A.domain_shape, '| range_shape (y):', A.range_shape)

# y = A(x) + n,  n ~ N(0, sigma^2)
sigma = 1.0
noiser = GaussianNoise(sigma=sigma)
with torch.no_grad():
    y_clean = A(x_true)          # 正问题：A(x)，得到 sinogram
    y = noiser(y_clean)          # 加噪声

show([x_true, y_clean, y],
     ['x  (ground truth)', 'A(x)  clean sinogram', 'y = A(x)+n'],
     sino_idx=(1,2))
print('sinogram 形状 (1,1,角度数,探测器数):', tuple(y.shape))

<a id="sec2"></a>
## 2. 病态性（ill-posedness）

### 2.1 Hadamard 三条件

Hadamard 称一个问题**良定（well-posed）**当且仅当：
1. **存在性**：解存在；
2. **唯一性**：解唯一；
3. **稳定性**：解**连续依赖**于数据（数据小扰动 ⟹ 解小扰动）。

反问题几乎总是**违反其中至少一条**——典型是第 3 条（稳定性）。这就是「病态」。

### 2.2 用 SVD 看病态（你最熟的语言）

线性情形 $y=Ax$，对 $A$ 做奇异值分解 $A=U\Sigma V^{\top}$，$\sigma_1\ge\sigma_2\ge\cdots\ge 0$。形式解：

$$x = A^{+}y = \sum_i \frac{u_i^{\top}y}{\sigma_i}\,v_i .$$

若数据含噪 $y=y^*+n$：

$$\hat{x} = \sum_i \frac{u_i^{\top}y^*}{\sigma_i}v_i \;+\; \underbrace{\sum_i \frac{u_i^{\top}n}{\sigma_i}v_i}_{\text{噪声被 } 1/\sigma_i \text{ 放大}} .$$

- 当 $\sigma_i \to 0$（小奇异值，对应高频 / 细节方向），$1/\sigma_i\to\infty$，**噪声被爆炸式放大**。
- **条件数** $\kappa(A)=\sigma_1/\sigma_{\min}$ 很大 ⟹ 病态。CT 在稀疏角 / 限角下 $\sigma_{\min}$ 极小。
- **欠定**（测量数 < 未知数，如稀疏角 CT、欠采样 MRI）⟹ 解不唯一（$A$ 有非平凡零空间 $\Rightarrow$ 违反唯一性）。

> **结论**：直接算 $A^{+}y$（朴素求逆）会把噪声放大到淹没信号。**解药 = 正则化**（第 4 节）：要么抑制小奇异值方向（Tikhonov / 截断 SVD / 早停），要么注入先验（TV / 稀疏 / 学习型先验）。

### 2.3 数值演示：噪声放大

下面对比：无噪 vs 有噪时，朴素重建（这里用 FBP，CT 里的解析反演 $\approx A^{+}$）的差别。

In [ ]:
from inv_framework.solvers.classical import FBPSolver

# 无噪测量 vs 有噪测量
with torch.no_grad():
    y_noisefree = A(x_true)
    y_noisy     = GaussianNoise(sigma=1.0)(y_noisefree)

rec_clean = FBPSolver().solve(y_noisefree, A).clamp(0,1)   # x ≈ A^+ y*（无噪）
rec_noisy = FBPSolver().solve(y_noisy,     A).clamp(0,1)   # x ≈ A^+ (y*+n)（有噪）

show([x_true, rec_clean, rec_noisy],
     ['x', 'FBP (clean) ~ A+ y*', 'FBP (noisy) ~ A+(y*+n)  <- noise blow-up'])
print('观察：右图高频噪声被小奇异值方向放大，这正是病态性的表现。')

### 2.4 稀疏角：违反唯一性

减少投影角 = 减少方程数 ⟹ $A$ 出现大零空间 ⟹ 无穷多个 $x$ 拟合同一个 $y$。朴素反演会出现**条纹伪影（streak artifacts）**。

In [ ]:
for n_ang in [90, 30, 10]:
    A_k = ParallelBeamRadon2D(image_size=N, num_angles=n_ang, device=DEVICE)
    with torch.no_grad():
        y_k = A_k(x_true)
    rec_k = FBPSolver().solve(y_k, A_k).clamp(0,1)
    show([rec_k], [f'FBP, {n_ang} angles'], figsize=(3,3.2))
print('角度越少 → 零空间越大 → 条纹伪影越重（欠定 / 违反唯一性）。')

<a id="sec3"></a>
## 3. 应用场景与前向模型

反问题的「灵魂」在于 $\mathcal{A}$。下面给出常见场景的数学前向模型，并指出在框架里该写成哪种算子。

### 3.1 线性反问题（$\mathcal{A}$ 线性，含 `adjoint`）

`LinearOperator` 比 `ForwardOperator` 多两条接口，对应线性代数里的转置与投影：

```python
class LinearOperator(ForwardOperator):
    def adjoint(self, y): ...                 # A^T y（伴随 / 转置）
    def pseudo_inverse(self, y, **kw): ...     # 默认用 Landweber 近似 A^+
    def project(self, x, y, step_size=None):   # 一步数据一致性投影
        return x - step_size * self.adjoint(self.forward(x) - y)
```

| 模态 | 前向模型 $\mathcal{A}$ | 公式 | 噪声 | 框架算子 |
|---|---|---|---|---|
| **CT**（计算机断层） | Radon 变换（线积分） | $(\mathcal{A}x)(\theta,s)=\int_{L_{\theta,s}} x\,d\ell$ | 泊松（光子） | ✅ `ParallelBeamRadon2D`（内置） |
| **MRI**（核磁） | 欠采样傅里叶 | $y=\mathcal{P}_\Omega \mathcal{F} x$ | 复高斯 | 自写 `LinearOperator`（见 §7） |
| **PET**（正电子) | 系统矩阵（响应线） | $y=Px,\ y_i\sim\mathrm{Poisson}((Px)_i)$ | 泊松 | 自写 `LinearOperator` + `PoissonLogDomainNoise` |
| **去模糊** | 卷积 | $y=k * x$ | 高斯 | 自写（FFT 实现 `forward/adjoint`） |
| **超分辨** | 模糊 + 下采样 | $y=S\,(k*x)$ | 高斯 | 自写 |

**伴随的意义**：$\langle \mathcal{A}x, y\rangle = \langle x, \mathcal{A}^{\top}y\rangle$。在 CT 里 $\mathcal{A}^{\top}$ 就是**反投影（back-projection）**——把 sinogram 沿射线「涂抹」回图像域。

> ⚠️ **本框架的核心工程约定**（必须理解）：`LinearOperator` 子类要保证 **`forward` 的 autograd backward 恰好等于 `adjoint`**。这样「基于自动微分的求解器」与「显式调用 `A^T` 的求解器」对 $\mathcal{A}^{\top}$ 的理解一致。`ParallelBeamRadon2D` 用自定义 `torch.autograd.Function` 强制成立（`tests/test_radon_adjoint.py` 验证相对误差 = 0）。注意这指的是 *autograd-伴随一致性*，不一定等于离散网格上的精确转置恒等式。

### 3.2 非线性反问题（$\mathcal{A}$ 非线性，无 `adjoint`，只继承 `ForwardOperator`）

很多 PDE 反演里，「介质参数 $x$ ↦ 观测 $y$」这个映射本身是**非线性**的（即使控制方程对波场线性）。这类问题**没有简单的解析 $\mathcal{A}^{\top}$**，但只要 `forward` 可微，就能用 DIP / INR / DPS 求解。

#### (a) 全波形反演 FWI / USCT — Helmholtz 方程
频域声波满足 Helmholtz 方程，未知量是**声速场 $c(\mathbf{r})$**（或慢度 $m=1/c^2$）：

$$\Big(\nabla^2 + \frac{\omega^2}{c(\mathbf{r})^2}\Big)\,u(\mathbf{r};\omega) = -s(\mathbf{r};\omega).$$

前向算子：给定 $c$，解 PDE 得到波场 $u$，再在接收器位置 $\{\mathbf{r}_j\}$ 采样：

$$\mathcal{A}(c) = \big\{\,u(\mathbf{r}_j;\omega)\,\big\}_{j,\omega}, \qquad u = \big(\nabla^2 + \omega^2/c^2\big)^{-1}(-s).$$

$c\mapsto u$ 经过一次 PDE 求逆，是**强非线性**的 ⟹ 必须用迭代 / 学习型方法（FWI 本质是 $\min_c \|\mathcal{A}(c)-y\|^2 + \lambda R(c)$）。

#### (b) 走时层析 — Eikonal 方程
只用初至走时（first-arrival travel time）时，走时 $T(\mathbf{r})$ 满足程函方程：

$$|\nabla T(\mathbf{r})|^2 = \frac{1}{c(\mathbf{r})^2},\qquad T(\text{source})=0.$$

前向算子 $\mathcal{A}(c)=\{T(\mathbf{r}_j)\}_j$（在接收器读走时），同样**非线性**于 $c$。

> 这些就是你们组 USCT / FWI 的数学骨架。在框架里，它们都写成
> ```python
> class HelmholtzFWI(ForwardOperator):
>     def forward(self, c):
>         u = solve_helmholtz(c, self.sources, self.omega)   # 可微 PDE solver
>         return sample_at_receivers(u, self.recv)
> ```
> 只要 `solve_helmholtz` 用可微算子（如可微有限差分 / CBS）实现，DIP / INR / DPS 直接可用，**求解器一行都不用改**。

#### (c) 其它非线性例子
- **相位恢复（phase retrieval）**：$y=|\mathcal{F}x|^2$（只测幅度，丢相位）。
- **饱和探测**：$y=\tanh(\alpha\,\mathcal{A}_{\text{lin}}x)$（探测器饱和），第 6 节会跑它的 demo。
- **盲去模糊**：核 $k$ 与 $x$ 同时未知，$y=k*x$ 对 $(k,x)$ 双线性、对整体非线性。

<a id="sec4"></a>
## 4. 迭代优化方法与正则化（重点：和你的优化背景直接对接）

### 4.1 变分 / 贝叶斯统一框架

把反问题写成你熟悉的优化问题：

$$\boxed{\;\hat{x} = \arg\min_{x}\; \underbrace{D\big(\mathcal{A}(x),\,y\big)}_{\text{数据保真}} \;+\; \lambda\, \underbrace{R(x)}_{\text{正则 / 先验}}\;}$$

- $D$ 由噪声决定：高斯 ⟹ $D=\tfrac12\|\mathcal{A}(x)-y\|_2^2$；泊松 ⟹ KL 散度。
- $R$ 注入先验知识，治病态。$\lambda$ 平衡两者。
- **贝叶斯视角（MAP）**：$\hat{x}=\arg\max_x p(x\mid y)=\arg\min_x[-\log p(y\mid x)-\log p(x)]$，于是 $D=-\log p(y|x)$（似然），$R=-\log p(x)$（先验）。

> 框架里：**`A` 决定 $D$，`Solver`（及其先验）决定 $R$。** Agent 的任务就是「给定 A，挑一个能提供好 $R$ 的 solver」。

### 4.2 经典迭代法（线性，显式用 $A^{\top}$）—— `solvers/classical.py`

**(1) Landweber = 对 $\tfrac12\|Ax-y\|^2$ 做梯度下降。** 梯度为 $A^{\top}(Ax-y)$：

$$x_{k+1} = x_k - \alpha\,A^{\top}(Ax_k - y).$$

对应代码 `landweber(...)`，以及 `LinearOperator.project()` 正是「一步 Landweber」：

```python
def project(self, x, y, step_size=None):
    residual = self.forward(x) - y        # A x - y
    return x - step_size * self.adjoint(residual)   # x - α Aᵀ(Ax-y)
```

**(2) SIRT** = 带行/列归一化（预条件）的 Landweber，收敛更快更稳：

$$x_{k+1} = x_k - C\,A^{\top}\big(R\,(Ax_k - y)\big),\quad R=\mathrm{diag}(1/\textstyle\sum_{\text{col}}A),\ C=\mathrm{diag}(1/\textstyle\sum_{\text{row}}A).$$

对应 `sirt(...)`，代码里 `R`、`C` 就是对 $A\mathbf{1}$、$A^{\top}\mathbf{1}$ 取倒数。

**(3) FBP** = CT 专属解析反演：先对 sinogram 做 **Ram–Lak 滤波**（补偿 $1/|\nu|$ 频率加权），再反投影 $A^{\top}$：

$$\hat{x} = A^{\top}\big(h_{\text{RamLak}} * y\big)\cdot\frac{\pi}{n_{\text{angles}}}.$$

> ⚠️ FBP 只对 CT 类算子有意义（它假设了 Radon 几何）。即便你的新算子是 `LinearOperator`，FBP 也未必适用——SIRT / Landweber 才是通用的线性求解器。

**正则化在经典法里以两种形式出现：**
- **早停（early stopping）= 隐式正则**。Landweber/SIRT 先恢复大奇异值（低频）方向，后恢复小奇异值（高频 / 噪声）方向 ⟹ 存在「半收敛（semi-convergence）」：迭代数适中时误差最小，过多则噪声涌入。下面 demo 会画出这条曲线。
- **盒约束（box constraint）**：`min_value` / `max_value` 把 $x$ 投影到 $[a,b]$（如衰减系数非负）。这是把 $R(x)=\iota_{[a,b]}(x)$（指示函数）加进去。

### 4.3 显式正则项 $R(x)$ 一览（数学系最关心的部分）

| 正则 $R(x)$ | 公式 | 偏好的解 | 物理含义 |
|---|---|---|---|
| **Tikhonov / L2** | $\|x\|_2^2$ 或 $\|\Gamma x\|_2^2$ | 能量小、光滑 | 抑制小奇异值方向（≈ 维纳滤波） |
| **总变差 TV** | $\|\nabla x\|_1=\sum\sqrt{x_{i+1,j}-x_{ij})^2+\cdots}$ | 分片常数、保边 | 图像分块均匀（CT 标配） |
| **稀疏 / L1** | $\|\Psi x\|_1$（$\Psi$=小波等） | 变换域稀疏 | 压缩感知（MRI 标配） |
| **盒 / 指示** | $\iota_{[a,b]}(x)$ | 物理可行域 | 非负性、上下界 |
| **学习型先验** | $-\log p_\theta(x)$ | 像「真实图像」 | DIP / INR / 扩散（下节） |

求解上述带 $R$ 的问题常用 **PGD / ISTA / FISTA / ADMM / Plug-and-Play**——全都是你熟的一阶法，框架的 `project()` 给了写 PnP / PGD 的积木（数据步 + 先验近端步交替）。

### 4.4 学习型先验：DIP / INR / 扩散（框架的 general 求解器）

当 $\mathcal{A}$ 非线性、或想用「数据驱动的先验」时，用下面三类。它们只要求 `A.forward` 可微（**不需要 `adjoint`**），所以**线性、非线性算子通吃**。

**(1) Deep Image Prior (DIP)** — `solvers/dip.py`。把 $x$ 重参数化为一个 CNN 的输出 $x=f_\theta(z)$（$z$ 固定随机），优化网络权重 $\theta$：

$$\min_\theta \;\big\|\,\mathcal{A}\big(f_\theta(z)\big) - y\,\big\|_2^2 .$$

CNN 结构本身就是隐式先验（先拟合低频自然结构、后拟合噪声 ⟹ 早停正则）。核心循环就是普通的 Adam：
```python
x_est = model(z); loss = ((A.forward(x_est) - y)**2).mean()
loss.backward(); optim.step()      # 对 θ 求梯度，autograd 自动穿过 A
```

**(2) Implicit Neural Representation (INR / SIREN)** — `solvers/inr.py`。把图像表示成坐标函数 $x(\mathbf{p})=f_\theta(\mathbf{p})$（$\mathbf{p}$ 是像素坐标），同样优化 $\theta$ 使 $\mathcal{A}(f_\theta)\approx y$。先验来自网络对坐标的平滑/周期性归纳偏置。

**(3) 扩散后验采样 DPS** — `solvers/diffusion/`。用一个**预训练的扩散先验**（学到 $\nabla_x\log p(x)$）做贝叶斯后验采样，从 $p(x\mid y)\propto p(y\mid x)\,p(x)$ 里采样。反向每一步：

1. 网络预测噪声 $\hat\epsilon=\epsilon_\theta(x_t,t)$；
2. **Tweedie 公式**估计干净图 $\hat{x}_0=\dfrac{x_t-\sqrt{1-\bar\alpha_t}\,\hat\epsilon}{\sqrt{\bar\alpha_t}}$（代码 `predict_x0_from_eps`）；
3. 先做一步无条件反扩散 $x_{t-1}'$（代码 `schedule.step` 返回 `(x_prev, x_0_pred)`）；
4. **数据一致性引导**（DPS 梯度），把测量信息注入：
$$x_{t-1} = x_{t-1}' - \zeta\,\nabla_{x_t}\big\|\,y-\mathcal{A}(\hat{x}_0)\,\big\|_2 .$$

第 4 步对应 `PosteriorSampling.apply`：
```python
class PosteriorSampling(ConditioningMethod):       # DPS, Chung et al. 2023
    def apply(self, x_t, x_0_hat, measurement, x_prev, **kw):
        diff = measurement - self.operator.forward(x_0_hat)   # y - A(x̂₀)
        residual = torch.linalg.norm(diff)
        grad, = torch.autograd.grad(residual, x_prev)         # ∇ 关于 x_t
        return x_t - self.scale * grad, residual.detach()     # ζ = self.scale
```

两个可替换 hook（都是 ABC，方便你扩展别的算法 MCG / PiGDM / RED）：
- `NoiseSchedule`：$\bar\alpha_t$ 调度与一步反扩散（内置 `VPSchedule`；或用 `DiffusersScheduleAdapter` 包 HuggingFace scheduler）；
- `ConditioningMethod`：测量引导项（内置 `PosteriorSampling`=DPS）。

> DPS 需要预训练好的 $\epsilon_\theta$，故本教程不在线训练，仅给接口骨架（见 §7）。训练脚本见 `examples/train_diffusion_for_dps.py`。

### 4.5 demo：半收敛曲线（早停 = 隐式正则）

下面在有噪 CT 上跑 Landweber，画「迭代数 vs PSNR」，亲眼看到误差先降后升。

In [ ]:
from inv_framework.solvers.classical import landweber
from inv_framework.utils.metrics import psnr

A2 = ParallelBeamRadon2D(image_size=N, num_angles=40, device=DEVICE)
with torch.no_grad():
    y2 = GaussianNoise(sigma=1.0)(A2(x_true))

x_k = torch.zeros(1,1,N,N, device=DEVICE)
step = 2e-3
iters, psnrs = [], []
for k in range(1, 121):
    # 手动跑一步 Landweber： x <- x - α Aᵀ(Ax - y)  （等价 A2.project）
    x_k = A2.project(x_k, y2, step_size=step).clamp(0,1)
    if k % 5 == 0:
        iters.append(k); psnrs.append(psnr(x_k, x_true).item())

best = iters[int(torch.tensor(psnrs).argmax())]
plt.figure(figsize=(5,3.2))
plt.plot(iters, psnrs, 'o-'); plt.axvline(best, ls='--', c='r')
plt.xlabel('Landweber iterations'); plt.ylabel('PSNR (dB)')
plt.title(f'semi-convergence: best early-stop ~ {best}'); plt.grid(alpha=.3); plt.show()
print('PSNR 先升后降 → 过度迭代把噪声（小奇异值方向）也拟合进来了。')

### 4.6 demo：学习型先验 DIP / INR（非线性也能用的通用解法）

> CPU 上用较少迭代演示机制（PSNR 不追求最优）。GPU + 更多迭代可显著提升。

In [ ]:
from inv_framework.solvers.dip import DIPSolver
from inv_framework.solvers.inr import INRSolver
from inv_framework.solvers.classical import SIRTSolver
from inv_framework.utils.metrics import psnr, ssim

A3 = ParallelBeamRadon2D(image_size=N, num_angles=30, device=DEVICE)
with torch.no_grad():
    y3 = GaussianNoise(sigma=1.0)(A3(x_true))

recs = {'Ground truth': x_true}
recs['FBP']  = FBPSolver().solve(y3, A3).clamp(0,1)
recs['SIRT'] = SIRTSolver(num_iterations=80, min_value=0, max_value=1).solve(y3, A3)
recs['DIP']  = DIPSolver(num_iterations=300, lr=1e-2).solve(y3, A3).clamp(0,1)   # CPU 友好
recs['INR']  = INRSolver(num_iterations=300, lr=1e-3,
                         hidden_features=128, hidden_layers=3).solve(y3, A3).clamp(0,1)

show(list(recs.values()), list(recs.keys()))
print(f"{'Method':12s}{'PSNR(dB)':>10s}{'SSIM':>8s}")
for k,v in recs.items():
    if k=='Ground truth': continue
    print(f"{k:12s}{psnr(v,x_true).item():10.2f}{ssim(v,x_true).item():8.3f}")

<a id="sec5"></a>
## 5. 评估指标 —— `utils/metrics.py`

重建好坏需要量化。框架内置两个**全参考（full-reference）**指标（需要 ground-truth $x^*$）。

**(1) PSNR（峰值信噪比，越大越好，dB）：**
$$\mathrm{MSE}=\frac1n\|\hat{x}-x^*\|_2^2,\qquad \mathrm{PSNR}=10\log_{10}\frac{\text{MAX}^2}{\mathrm{MSE}}.$$
代码 `psnr(pred, target, data_range=1.0)`，`data_range`=$\text{MAX}$（图像归一化到 $[0,1]$ 时取 1）。

**(2) SSIM（结构相似性，$\in[-1,1]$，越大越好）：** 同时比较亮度、对比度、结构：
$$\mathrm{SSIM}(x,y)=\frac{(2\mu_x\mu_y+C_1)(2\sigma_{xy}+C_2)}{(\mu_x^2+\mu_y^2+C_1)(\sigma_x^2+\sigma_y^2+C_2)} .$$
代码 `ssim(pred, target, data_range=1.0)`，用高斯窗口在局部统计 $\mu,\sigma$。PSNR 对逐像素误差敏感但忽略结构；SSIM 更贴近人眼，两者常一起报。

**(3) 数据残差（无 ground-truth 时唯一能看的量）：**
$$\text{relative residual} = \frac{\|\mathcal{A}(\hat{x})-y\|_2}{\|y\|_2}.$$
真实任务里没有 $x^*$，只能看这个 + 先验合理性。注意：残差小 ≠ 重建对（病态！欠定时残差可为 0 而 $x$ 全错），所以才需要正则。

In [ ]:
from inv_framework.utils.metrics import psnr, ssim

xhat = recs['SIRT']
print('PSNR =', round(psnr(xhat, x_true).item(), 2), 'dB')
print('SSIM =', round(ssim(xhat, x_true).item(), 3))
with torch.no_grad():
    rel_res = (A3(xhat) - y3).norm() / y3.norm()
print('relative data residual =', round(rel_res.item(), 4), '  (无 GT 时唯一可观测量)')

<a id="sec6"></a>
## 6. 非线性 demo 与求解器分派（Agent 的核心调度逻辑）

这一节是给 **Agent 框架前端**最关键的一节：**算子类型决定了哪些求解器合法**。框架已经把这条规则硬编码进类型系统。

### 6.1 分派规则

| 求解器 | 接受的算子类型 | 需要 `adjoint`？ | 代码里的拒绝机制 |
|---|---|---|---|
| `FBPSolver` / `SIRTSolver` / `LandweberSolver` | **必须 `LinearOperator`** | 是 | `_require_linear()` 抛 `TypeError` |
| `DIPSolver` / `INRSolver` / `DPSSolver` | **任意 `ForwardOperator`** | 否（只用 autograd） | 无限制 |

```python
def _require_linear(op, name):
    if not isinstance(op, LinearOperator):
        raise TypeError(f"{name} requires a LinearOperator ... Use DIPSolver/INRSolver/DPSSolver")
```

> **Agent 调度建议**：拿到一个 `operator` 后，用 `isinstance(op, LinearOperator)` 判一刀：
> - 是线性 → 可选 {经典法（快、可解释）, DIP, INR, DPS}；
> - 非线性 → 只能选 {DIP, INR, DPS}。
> 框架会在你选错时**主动报错并提示**，这正好可作为 Agent 的「工具前置校验 / 错误恢复」信号。

### 6.2 demo：饱和探测（非线性 CT）

前向 $\;y=\tanh\!\big(\alpha\,\mathcal{A}_{\text{Radon}}(x)/N\big)$，非线性 ⟹ FBP 应拒绝，DIP 仍可解。

In [ ]:
from inv_framework.operators.base import ForwardOperator

class SaturatedRadon2D(ForwardOperator):
    '''非线性前向：y = tanh(alpha * Radon(x) / N)。只继承 ForwardOperator，无 adjoint。'''
    def __init__(self, image_size, num_angles, alpha=2.0, device='cpu'):
        self.linear = ParallelBeamRadon2D(image_size, num_angles, device=device)
        self.alpha = float(alpha); self.image_size = int(image_size)
        self.domain_shape = self.linear.domain_shape
        self.range_shape  = self.linear.range_shape
    def forward(self, x):
        sino = self.linear.forward(x) / self.image_size      # autograd 可穿过
        return torch.tanh(self.alpha * sino)

op_nl = SaturatedRadon2D(N, num_angles=60, alpha=2.0, device=DEVICE)
with torch.no_grad():
    y_nl = GaussianNoise(sigma=0.02)(op_nl(x_true))

# 1) 经典法应当拒绝非线性算子
print('--- 把 FBP 用在非线性算子上（预期抛 TypeError）---')
try:
    FBPSolver().solve(y_nl, op_nl)
except TypeError as e:
    print('  caught TypeError:', str(e)[:90], '...')

# 2) DIP 不需要 adjoint，照样工作
x_nl = DIPSolver(num_iterations=400, lr=1e-2).solve(y_nl, op_nl).clamp(0,1)
show([x_true, y_nl, x_nl], ['x', 'y (saturated sinogram)', 'DIP reconstruction'], sino_idx=(1,))
print('DIP PSNR =', round(psnr(x_nl, x_true).item(), 2), 'dB  ——非线性也能解')

<a id="sec7"></a>
## 7. 暴露给前端：如何扩展（你们真正要写的东西）

框架的扩展点正是你们 Agent 要「注册成工具 / 可调度组件」的对象。**加新反问题不需要改任何求解器代码。**

### 7.1 四类扩展点

| 想加什么 | 继承谁 | 必须实现 | 文件参考 |
|---|---|---|---|
| 新**线性**算子（MRI / 去模糊 / inpainting） | `LinearOperator` | `forward` + `adjoint` | `operators/base.py` |
| 新**非线性**算子（FWI / 相位恢复） | `ForwardOperator` | `forward`（可微即可） | 同上 |
| 新**噪声 / 似然** | `NoiseModel` | `forward` | `operators/noise.py` |
| 新**扩散引导**（MCG / PiGDM / RED） | `ConditioningMethod` | `apply` | `solvers/diffusion/conditioning.py` |
| 新**求解器** | `InverseProblemSolver` | `solve(measurement, operator, **kw)` | `solvers/base.py` |

### 7.2 demo：从零写一个新线性算子（图像修复 inpainting）

前向 $\;y = M\odot x\;$（$M$ 是 0/1 掩码，丢失部分像素）。这是个**对角线性算子**，自伴随 $\mathcal{A}^{\top}=\mathcal{A}$（$M^\top M=M$）。由于是逐元素乘，`forward` 的 autograd backward 自动等于 `adjoint`，无需自定义 `autograd.Function`。

写完直接喂给 `SIRTSolver` / `DIPSolver`——**求解器零改动**。

In [ ]:
from inv_framework.operators.base import LinearOperator

class Inpainting2D(LinearOperator):
    '''y = M ⊙ x。线性、自伴随。展示『加新算子，求解器不改』。'''
    def __init__(self, image_size, mask, device='cpu'):
        self.mask = mask.to(device)                       # (1,1,H,W) ∈ {0,1}
        self.domain_shape = (1, image_size, image_size)
        self.range_shape  = (1, image_size, image_size)
    def forward(self, x):  return x * self.mask           # A x
    def adjoint(self, y):  return y * self.mask           # Aᵀ y = A y（自伴随）

# 随机丢 60% 像素
mask = (torch.rand(1,1,N,N, device=DEVICE) > 0.6).float()
A_inp = Inpainting2D(N, mask, device=DEVICE)
with torch.no_grad():
    y_inp = A_inp(x_true)                                  # 无噪，纯掩码

# 同一批求解器，完全不改：SIRT（线性）+ DIP（通用）
rec_sirt = SIRTSolver(num_iterations=60, min_value=0, max_value=1).solve(y_inp, A_inp)
rec_dip  = DIPSolver(num_iterations=400, lr=1e-2).solve(y_inp, A_inp).clamp(0,1)

show([x_true, y_inp, rec_sirt, rec_dip],
     ['x', 'y = M*x (60% missing)', 'SIRT', 'DIP (prior fills in)'])
print('SIRT 只能复制可见像素（无先验）；DIP 用网络先验把缺失区域补出来。')
print('DIP PSNR =', round(psnr(rec_dip, x_true).item(),2), 'dB')

### 7.3 MRI（欠采样傅里叶）算子骨架

实际你们要接 MRI 时，照搬 README 的 `LinearOperator` 写法即可（复数拆成 real/imag 两通道）：

```python
class UnderSampledFourier2D(LinearOperator):
    def __init__(self, image_size, mask):           # mask: (H,W) bool, k-space 采样图案
        self.mask = mask
        self.domain_shape = (2, image_size, image_size)   # x: real+imag
        self.range_shape  = (2, image_size, image_size)
    def forward(self, x):                            # y = P_Ω F x
        z = torch.complex(x[:,0], x[:,1])
        Y = torch.fft.fft2(z) * self.mask
        return torch.stack([Y.real, Y.imag], dim=1)
    def adjoint(self, y):                            # Aᵀ = Fᴴ P_Ω
        Y = torch.complex(y[:,0], y[:,1]) * self.mask
        z = torch.fft.ifft2(Y) * Y.numel()
        return torch.stack([z.real, z.imag], dim=1)
```

### 7.4 扩散引导（自定义 ConditioningMethod）骨架

想实现 DPS 之外的引导（如 MCG = 梯度 + 投影），继承 `ConditioningMethod`：

```python
from inv_framework.solvers.diffusion.conditioning import ConditioningMethod
from inv_framework.solvers.diffusion.dps import DPSSolver

class MCG(ConditioningMethod):
    def apply(self, x_t, x_0_hat, measurement, x_prev, **kw):
        # 1) DPS 梯度项
        diff = measurement - self.operator.forward(x_0_hat)
        grad, = torch.autograd.grad(torch.linalg.norm(diff), x_prev)
        x_t = x_t - self.scale * grad
        # 2) 若是线性算子，再加一步数据一致性投影 A.project(...)
        return x_t, diff.norm().detach()

# 用法（需预训练 eps_theta）：
# solver = DPSSolver(model=eps_theta, conditioning=MCG(operator, scale=1.0))
# x_rec = solver.solve(y, operator)
```

> **Agent 框架视角的总结**：把 `ForwardOperator` 子类注册成「问题类型」，`NoiseModel` 注册成「噪声/似然」，`InverseProblemSolver` 子类注册成「可调度算法」。Agent 的决策 = (1) 根据物理选 operator；(2) 根据噪声选 noiser；(3) 根据 `isinstance(op, LinearOperator)` + 速度/质量需求选 solver。三者用统一的 `solve(y, op)` 接口拼起来即可。

<a id="sec8"></a>
## 8. 速查表：公式 ↔ 代码

| 数学 | 含义 | 代码接口 |
|---|---|---|
| $y=\mathcal{A}(x)+n$ | 反问题总式 | `y = noiser(operator(x))` |
| $\mathcal{A}(x)$ | 前向算子 | `ForwardOperator.forward(x)` / `op(x)` |
| $\mathcal{A}^{\top}y$ | 伴随 / 反投影 | `LinearOperator.adjoint(y)` |
| $\mathcal{A}^{+}y$ | 伪逆 | `LinearOperator.pseudo_inverse(y)` |
| $x-\alpha\mathcal{A}^{\top}(\mathcal{A}x-y)$ | 一步梯度/投影 | `LinearOperator.project(x,y,step_size)` |
| $n\sim\mathcal{N}(0,\sigma^2)$ | 高斯似然 | `GaussianNoise(sigma)` |
| $\mathrm{Poisson}(I_0e^{-\mu x})$ | 泊松似然 | `PoissonLogDomainNoise(photon_count)` |
| $x_{k+1}=x_k-\alpha A^{\top}(Ax_k-y)$ | Landweber | `landweber(...)` / `LandweberSolver` |
| SIRT 加权迭代 | 预条件迭代 | `sirt(...)` / `SIRTSolver` |
| $A^{\top}(h_{\text{RamLak}}*y)$ | FBP | `fbp(...)` / `FBPSolver` |
| $\min_\theta\|\mathcal{A}(f_\theta(z))-y\|^2$ | DIP | `DIPSolver` |
| $x=f_\theta(\mathbf{p})$ | INR | `INRSolver` |
| $\hat{x}_0=(x_t-\sqrt{1-\bar\alpha_t}\hat\epsilon)/\sqrt{\bar\alpha_t}$ | Tweedie | `NoiseSchedule.predict_x0_from_eps` |
| 一步反扩散 → $(x_{t-1},\hat x_0)$ | 采样步 | `NoiseSchedule.step(eps,t,x_t,eta)` |
| $x_{t-1}'-\zeta\nabla\|y-\mathcal{A}(\hat x_0)\|$ | DPS 引导 | `PosteriorSampling.apply(...)` |
| $\arg\min_x D+\lambda R$ | 变分总式 | 选 `operator`（定 $D$）+ `solver`（定 $R$） |
| $10\log_{10}(\text{MAX}^2/\text{MSE})$ | PSNR | `metrics.psnr` |
| SSIM | 结构相似 | `metrics.ssim` |

### 关键文件地图
- `operators/base.py` — `ForwardOperator` / `LinearOperator`（两层抽象）
- `operators/noise.py` — 噪声 / 似然
- `operators/ct/radon_torch.py` — CT 内置算子（autograd-伴随一致性的范本）
- `solvers/classical.py` — FBP / SIRT / Landweber（线性，显式 `A^T`）
- `solvers/dip.py`, `solvers/inr.py` — 学习型先验（通用）
- `solvers/diffusion/` — `scheduler.py` / `conditioning.py` / `dps.py`（扩散，diffusers 兼容）
- `utils/metrics.py` — PSNR / SSIM
- `examples/` — `ct_demo.py` / `nonlinear_demo.py` / `train_diffusion_for_dps.py`
- `tests/` — `test_radon_adjoint.py`（伴随一致性 + FBP）/ `test_nonlinear_operator.py`

### 下一步
- 跑通仓库自带完整 demo：`python examples/ct_demo.py --image-size 128 --num-angles 30`
- 训练扩散先验后试 DPS：`python examples/train_diffusion_for_dps.py` → `python examples/ct_demo.py --dps-checkpoint ...`
- 给 Agent 注册新算子时，参考 §7 的 `Inpainting2D` / `UnderSampledFourier2D` 模板。

> 有任何「这条公式对应哪个函数」的疑问，回到第 8 节速查表，或直接读对应文件——每个文件头部 docstring 都写了它实现的数学。